### 嵌入模型

* embed_documents
* embed_query
* 嵌入缓存
* 使用国产的嵌入模型

#### embed_documents

### Local embedding model
Use the installed HuggingFaceEmbeddings integration with BAAI/bge-small-en-v1.5 on CPU. No API key or proxy endpoint is required. The first run downloads the model; subsequent runs use the local cache.
For Chinese text, change MODEL_NAME to BAAI/bge-small-zh-v1.5; for multilingual text, use BAAI/bge-m3 (more memory required).


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

# English model for the example texts below.
MODEL_NAME = "BAAI/bge-small-en-v1.5"

embeddings_model = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print(f"Local embedding model: {MODEL_NAME}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Local embedding model: BAAI/bge-small-en-v1.5


In [6]:
texts = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World!",
]
embeddings = embeddings_model.embed_documents(texts)
print(f"Documents: {len(embeddings)}, dimensions: {len(embeddings[0])}")
embeddings[0][:5]


Documents: 5, dimensions: 384


[-0.049872513860464096,
 -0.043909668922424316,
 0.07376978546380997,
 -0.021135905757546425,
 0.03296177461743355]

#### embed_query
用同一个向量模型把查询文本转换为向量。


In [7]:
query = "How do embeddings represent meaning?"
query_embedding = embeddings_model.embed_query(query)
len(query_embedding), query_embedding[:5]


(384,
 [-0.008298812434077263,
  0.013388076797127724,
  -0.005489172879606485,
  -0.013986382633447647,
  -0.013466215692460537])

#### 缓存嵌入结果

In [10]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

# 复用前面的本地向量模型
underlying_embeddings = embeddings_model

# 向量缓存保存到当前目录
store = LocalFileStore("./langchain_cache")
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace=underlying_embeddings.model_name.replace("/", "_") + "_normalized_",
    key_encoder="sha256",
)


In [11]:
# 查看已缓存的向量，首次运行可能是空列表
list(store.yield_keys())


[]

##### 读取文档并切分


In [12]:
# 使用项目里现有的 RAG 学习文档
raw_documents = TextLoader(
    "F:/workspace/PycharmProjects/langchain_learning/chapter03_rag/RAG学习.md",
    encoding="utf-8",
).load()

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=0,
)
documents = text_splitter.split_documents(raw_documents)
print("文本块数：", len(documents))


Created a chunk of size 1448, which is longer than the specified 1000


文本块数： 35


##### 创建 FAISS 向量存储
第一次计算向量并写入缓存，第二次读取缓存，通常会更快。


In [13]:
from time import perf_counter

start = perf_counter()
db = FAISS.from_documents(documents, cached_embedder)
print(f"首次创建耗时：{perf_counter() - start:.2f} 秒")
print("缓存数量：", len(list(store.yield_keys())))


首次创建耗时：3.99 秒
缓存数量： 35


In [14]:
start = perf_counter()
db = FAISS.from_documents(documents, cached_embedder)
print(f"复用缓存耗时：{perf_counter() - start:.2f} 秒")


复用缓存耗时：0.02 秒


In [15]:
# 检索最相关的两个文本块
results = db.similarity_search("What is RAG?", k=2)
for doc in results:
    print(doc.page_content)
    print("-" * 50)


**校正与边界：**有 RAG 不等于天然可追溯。还要保留元数据、生成正确引用，并验证引用确实支持对应说法。检索能力也不等于读取权限，私有数据仍需访问控制。

### 1.2 RAG、长上下文、微调分别解决什么问题

| 方案 | 主要机制 | 优先考虑的场景 | 主要代价与边界 |
| --- | --- | --- | --- |
| RAG | 运行时检索外部证据 | 知识更新频繁、资料多、需要出处 | 需要维护知识库，效果受检索与证据质量影响 |
| 长上下文 | 把较多资料直接放入输入 | 资料规模可控，需要同时阅读全文 | 输入成本、延迟和有效利用能力需要实测 |
| 微调 | 用训练数据更新模型参数或适配器 | 改善领域任务表现、表达习惯、指令遵循 | 训练、数据和回归成本，知识难精确更新与溯源 |
| Prompt／Few-Shot | 在上下文中给指令和示例 | 快速约束风格、格式、任务方法 | 效果与上下文预算有限，需评估稳定性 |

例如法律助手可以用微调改善文书表达，用 RAG 提供最新法条，用较长上下文处理同一案件的多份证据。也可能用通用模型加 RAG 就满足需求，不必默认再做微调。

**校正与边界：**

- 微调能够学习新增训练数据中的部分知识，但不会自动持续获取训练结束后的新事实，也不是精确维护事实数据库的可靠替代。
- 不存在“少于 1000 条一定不值得微调”“有上万条就必须微调”的统一门槛。质量、任务和基线决定收益。
- LoRA 是参数高效微调方法，常用于降低显存和训练成本；它不保证消除灾难性遗忘，也不保证与全参数微调等效。
- 结构化输出可以先用 Schema、约束解码、验证与重试解决，不必一开始就微调。
- Few-Shot 是上下文学习的一种用法，上下文也可以放事实资料；它与检索解决的环节不同，可以组合。

若确实需要微调，可以先由模型生成候选训练样本，再让领域专家审核答案、术语和格式，保留独立验证集。少量精标数据往往比大量错误样本更有价值，但不能据此承诺某个固定样本量必然有效；还要检查原有任务是否退化。

### 1.3 Agent 为什么需要 RAG，何时应该查询数据库

Agent 负责围绕目标决定和执行动作；RAG 可以作为它获取外部事实、长期记忆和背景知识的工具。

例如“帮我判断这位员工今年还能休几天”：
